# Gerar exemplos reais de match e não-match para o app

In [3]:
import pandas as pd
import joblib
import numpy as np
import json

# Caminhos
DATASET_PATH = '../output/dataset_unificado_balanceado.csv'
MODELO_PATH = '../output/modelo_match_xgb.joblib'
PREPROCESSADOR_PATH = '../output/preprocessador_xgb.joblib'
VETORIZADOR_PATH = '../output/vetorizador_sim_textual.joblib'

# Carregar artefatos
df = pd.read_csv(DATASET_PATH)
modelo = joblib.load(MODELO_PATH)
preprocessador = joblib.load(PREPROCESSADOR_PATH)
vetorizador_sim = joblib.load(VETORIZADOR_PATH)

# Recalcular similaridade com peso ajustado
df['requisitos_vaga'] = df['requisitos_vaga'].fillna('').str.lower()
df['cv_texto'] = df['cv_texto'].fillna('').str.lower()

req_vec = vetorizador_sim.transform(df['requisitos_vaga'])
cv_vec = vetorizador_sim.transform(df['cv_texto'])
df['sim_textual'] = np.array(req_vec.multiply(cv_vec).sum(axis=1)).ravel() * 0.3

# Recalcular features auxiliares (compatível com app)
df['match_nivel'] = (df['nivel_profissional_vaga'] == df['nivel_profissional_candidato']).astype(int)
df['match_profissional'] = df['match_nivel']
df['match_ingles'] = (df['nivel_ingles_vaga'] == df['nivel_ingles_candidato']).astype(int)
df['match_espanhol'] = (df['nivel_espanhol_vaga'] == df['nivel_espanhol_candidato']).astype(int)
df['match_local'] = (df['local_vaga'] == df['local_candidato']).astype(int)
df['match_academico'] = (df['nivel_academico_vaga'] == df['nivel_academico_candidato']).astype(int)

# Aplicar pré-processamento
df_modelo = df[preprocessador.feature_names_in_].copy()
X = preprocessador.transform(df_modelo)

# Fazer predição
df['prob'] = modelo.predict_proba(X)[:, 1]
df['predito'] = (df['prob'] >= 0.5).astype(int)

# Selecionar top 10 vagas com mais exemplos
top_vagas = df['vaga_id'].value_counts().head(10).index.tolist()
exemplos = []
for vaga_id in top_vagas:
    subset = df[df['vaga_id'] == vaga_id]
    match = subset[subset['predito'] == 1].head(1)
    no_match = subset[subset['predito'] == 0].head(1)
    if not match.empty:
        exemplos.append(match.assign(tipo_resultado='match'))
    if not no_match.empty:
        exemplos.append(no_match.assign(tipo_resultado='no_match'))

# Concatenar e exportar
exemplos = pd.concat(exemplos).reset_index(drop=True)
output_dict = {}
for _, row in exemplos.iterrows():
    vaga_id = row['vaga_id']
    tipo = row['tipo_resultado']
    if vaga_id not in output_dict:
        output_dict[vaga_id] = {}
    output_dict[vaga_id][tipo] = row.drop(['vaga_id', 'tipo_resultado']).to_dict()

with open('../output/exemplos_para_teste_app.json', 'w', encoding='utf-8') as f:
    json.dump(output_dict, f, ensure_ascii=False, indent=2)

print('Arquivo salvo: etl/output/exemplos_para_teste_app.json')

Arquivo salvo: etl/output/exemplos_para_teste_app.json
